# RAG Workflow for Geotechnical PDFs

We build a Retrieval-Augmented Generation (RAG) pipeline that indexes the geotechnical PDFs in `docs/05-llm/docs/`, stores embeddings in ChromaDB, and answers engineering questions with GPT-4o-mini plus citations.

### Learning goals
- Parse multi-document PDF corpora into token-aware chunks
- Persist embeddings locally with ChromaDB for fast iteration
- Craft prompts that combine retrieved context with deterministic calculations
- Quantify cost, latency, and validation checks required for productionization

In [ ]:
!pip install -q openai pydantic chromadb pypdf2 tiktoken python-dotenv

> **Colab note:** `chromadb`, `PyPDF2`, and the other dependencies install cleanly with the `pip` cell above, so this workflow runs unchanged on Google Colab (local paths simply need to point at the uploaded `docs/*.pdf` files).

In [ ]:
!wget https://raw.githubusercontent.com/kks32-courses/ai-geotech/refs/heads/main/docs/05-llm/docs.zip
!unzip docs.zip

In [ ]:
import json
import math
import os
from pathlib import Path
from typing import Dict, List

import chromadb
import matplotlib.pyplot as plt
import numpy as np
from dotenv import load_dotenv
from getpass import getpass
from openai import OpenAI
from PyPDF2 import PdfReader
import tiktoken

## API key and environment controls
`.env` is preferred, `getpass` is the fallback, and a commented placeholder is available for smoke tests. Errors are explicit if no key is found.

In [ ]:
def load_api_key(env_path: str = ".env") -> str:
    env_file = Path(env_path)
    if env_file.exists():
        load_dotenv(env_file)
    key = os.getenv("OPENAI_API_KEY")
    if not key:
        try:
            key = getpass("Enter your OpenAI API key: ").strip()
        except Exception as exc:
            raise RuntimeError("Unable to capture OPENAI_API_KEY via getpass.") from exc
    if not key:
        raise RuntimeError("OPENAI_API_KEY missing. Create a .env file or enter it interactively.")
    return key

# OPENAI_API_KEY = "sk-proj-example"  # Uncomment only for offline linting
OPENAI_API_KEY = load_api_key()
print(f"Loaded API key prefix: {OPENAI_API_KEY[:8]}******** (masked)")
client = OpenAI(api_key=OPENAI_API_KEY)

## Quick sanity check: basic GPT-4o-mini call
Before building the RAG stack, confirm that the OpenAI client works by requesting a concise greeting.

In [ ]:
hello_completion = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0.2,
    messages=[
        {"role": "system", "content": "You are a friendly geotechnical assistant."},
        {"role": "user", "content": "Greet the CE397 class in one sentence."},
    ],
)
print(hello_completion.choices[0].message.content)
if hello_completion.usage:
    print("Token usage -> input:", hello_completion.usage.prompt_tokens,
          "output:", hello_completion.usage.completion_tokens)

## Document inventory
Inspect the PDFs that will be ingested so we can cross-check naming conventions and sizes.

In [ ]:
PDF_DIR = Path("docs")
pdf_paths = sorted(PDF_DIR.glob("*.pdf"))
if not pdf_paths:
    raise FileNotFoundError("No PDFs found in docs/05-llm/docs. Populate the folder before running this notebook.")
for pdf in pdf_paths:
    size_mb = pdf.stat().st_size / (1024 * 1024)
    print(f"- {pdf.name}: {size_mb:.2f} MB")

## PDF parsing helpers
`PyPDF2` extracts text per page so that we can keep page numbers with every chunk for citations later.

In [ ]:
def load_pdf_pages(pdf_path: Path) -> List[Dict]:
    reader = PdfReader(str(pdf_path))
    pages = []
    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        pages.append({"page": page_number, "text": text})
    return pages


def preview_pdf(pdf_path: Path, n_chars: int = 600) -> str:
    pages = load_pdf_pages(pdf_path)
    combined = "".join(page["text"] for page in pages)
    return combined[:n_chars]

## Quick preview
Grab a short snippet so we can sanity-check encoding and confirm that the text (not just images) is available.

In [ ]:
print(preview_pdf(pdf_paths[0]))

## Chunking strategy
We chunk each PDF page using the `cl100k_base` tokenizer. Each chunk carries its token length, character length, originating page, and document name.

In [ ]:
ENCODER = tiktoken.get_encoding("cl100k_base")
CHUNK_SIZE = 520
CHUNK_OVERLAP = 80


def chunk_text(pages: List[Dict], chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[Dict]:
    chunks: List[Dict] = []
    for page in pages:
        tokens = ENCODER.encode(page["text"])
        start = 0
        while start < len(tokens):
            end = min(len(tokens), start + chunk_size)
            token_slice = tokens[start:end]
            text = ENCODER.decode(token_slice)
            chunks.append(
                {
                    "page": page["page"],
                    "text": text,
                    "token_length": len(token_slice),
                    "char_length": len(text),
                }
            )
            if end >= len(tokens):
                break
            start = max(0, end - overlap)
    return chunks

## Embedding configuration
We use `text-embedding-3-small` (good cost/performance) and persist vectors inside `docs/05-llm/chroma_db` for repeatability.

In [ ]:
EMBEDDING_MODEL = "text-embedding-3-small"
CHROMA_PATH = Path("docs/05-llm/chroma_db")
CHROMA_PATH.mkdir(parents=True, exist_ok=True)

chroma_client = chromadb.PersistentClient(path=str(CHROMA_PATH))
collection = chroma_client.get_or_create_collection(name="geotech_rag", metadata={"hnsw:space": "cosine"})
print("Chroma collection contains", collection.count(), "vectors before this run.")

## Build / refresh the vector store
Chunk each PDF, embed lazily in small batches, and `upsert` them into ChromaDB. Progress is printed per document.

In [ ]:
chunk_registry: List[Dict] = []

BATCH_SIZE = 32

def embed_batch(text_batch: List[str]) -> List[List[float]]:
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=text_batch)
    return [record.embedding for record in response.data]

for pdf in pdf_paths:
    pages = load_pdf_pages(pdf)
    chunks = chunk_text(pages)
    print(f"{pdf.name}: {len(pages)} pages -> {len(chunks)} chunks")
    payload = [
        {
            "id": f"{pdf.stem}-chunk-{idx}",
            "text": chunk["text"],
            "metadata": {
                "source": pdf.name,
                "page": chunk["page"],
                "token_length": chunk["token_length"],
            },
        }
        for idx, chunk in enumerate(chunks)
    ]
    for start in range(0, len(payload), BATCH_SIZE):
        batch = payload[start : start + BATCH_SIZE]
        embeddings = embed_batch([item["text"] for item in batch])
        collection.upsert(
            ids=[item["id"] for item in batch],
            embeddings=embeddings,
            metadatas=[item["metadata"] for item in batch],
            documents=[item["text"] for item in batch],
        )
    chunk_registry.extend(
        {
            "doc": item["metadata"]["source"],
            "page": item["metadata"]["page"],
            "tokens": item["metadata"]["token_length"],
            "id": item["id"],
        }
        for item in payload
    )
print("Total chunks indexed this session:", len(chunk_registry))

## Chunk statistics
We track chunk counts, tokens per chunk, and enforce the ~60 chunks / 100 pages target from the prompt.

In [ ]:
pages_per_doc = {pdf.name: len(load_pdf_pages(pdf)) for pdf in pdf_paths}
total_pages = sum(pages_per_doc.values())
total_chunks = len(chunk_registry)
chunks_per_100_pages = (total_chunks / total_pages * 100) if total_pages else 0
avg_tokens = np.mean([entry["tokens"] for entry in chunk_registry]) if chunk_registry else 0
print(json.dumps({
    "total_pages": total_pages,
    "total_chunks": total_chunks,
    "chunks_per_100_pages": round(chunks_per_100_pages, 1),
    "avg_tokens_per_chunk": round(float(avg_tokens), 1),
}, indent=2))

## Retrieval + generation helper
`query_rag` performs vector search, formats the context, and calls GPT-4o-mini with citations.

In [ ]:

from typing import Dict

def retrieve(query: str, k: int = 5) -> Dict:
    query_embedding = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=[query],
    ).data[0].embedding






def format_context(results: Dict) -> Dict:
    context_blocks = []
    citations = []
    if not results.get("documents"):
        return {"context": "", "citations": []}
    for idx, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
        label = f"Source {idx + 1}"
        snippet = doc.strip().replace("", " ")
        context_blocks.append(
            f"{label} ({meta['source']} p.{meta['page']}):{snippet}"
        )
        citations.append(f"{label}: {meta['source']} p.{meta['page']}")
    return {"context": "".join(context_blocks).strip(), "citations": citations}


def query_rag(question: str, k: int = 5) -> Dict:
    




    
    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.3,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a geotechnical engineer. Use the provided sources only, "
                    "show calculations when relevant, and cite sources as [Source i]."
                ),
            },
            {"role": "user", "content": user_content},
        ],
    )
    answer = completion.choices[0].message.content
    return {"answer": answer, "citations": formatted["citations"], "raw": results}


## Example engineering query
Ask for allowable bearing pressure guidance while forcing GPT-4o-mini to stay grounded in the retrieved sources.

In [ ]:
rag_question = "Summarize the recommended allowable bearing pressures and instrumentation notes for the foundation investigations."
rag_outputs = query_rag(rag_question, k=4)
print(rag_outputs["answer"])
print("Citations:")
for citation in rag_outputs["citations"]:
    print("-", citation)

## Targeted query: seismic considerations for the three-story building
With the vector store in place, we can now ask specifically about how the reports address seismic design for the three-story structure mentioned in `TERRACON_FINALV5.pdf`.

In [ ]:
seismic_question = "What were the seismic considerations for the proposed three-story building?"
seismic_outputs = query_rag(seismic_question, k=5)
print(seismic_outputs["answer"])
print("Citations:")
for citation in seismic_outputs["citations"]:
    print("-", citation)

## Cost estimation
Estimate the embedding cost for the current corpus and the marginal cost per RAG query. Assumptions follow the TASK_PROMPT guidance.

In [ ]:
EMBED_COST_PER_M_TOKEN = 0.02  # USD per 1M tokens for text-embedding-3-small
GENERATION_COST_PER_QUERY = 0.001  # Approximate per the prompt guidance

total_tokens_indexed = sum(entry["tokens"] for entry in chunk_registry)
embed_cost = total_tokens_indexed / 1_000_000 * EMBED_COST_PER_M_TOKEN
hundred_query_cost = 100 * GENERATION_COST_PER_QUERY
print(json.dumps({
    "tokens_indexed": total_tokens_indexed,
    "embedding_cost_usd": round(embed_cost, 4),
    "per_query_generation_cost_usd": GENERATION_COST_PER_QUERY,
    "hundred_query_cost_usd": round(hundred_query_cost, 3),
}, indent=2))

## Operational tests
Validate API key loading, Chroma persistence, chunk counts, and retrieval output structure.

In [ ]:
assert OPENAI_API_KEY, "API key failed to load"
assert CHROMA_PATH.exists(), "Chroma directory missing"
assert chunk_registry, "Chunk registry is empty; run the build cell first"
if rag_outputs:
    assert rag_outputs["citations"], "RAG output missing citations"
    assert "documents" in rag_outputs["raw"], "Raw retrieval lacks documents"
    print("All operational tests passed.")
else:
    print("Run the query cell above to populate rag_outputs before testing.")

## Chunk length distribution
Visualize the token length distribution to spot outliers (e.g., OCR issues or tables that need special handling).

In [ ]:
tokens = [entry["tokens"] for entry in chunk_registry]
plt.figure(figsize=(7, 4))
plt.hist(tokens, bins=20, color="#4c72b0", edgecolor="white")
plt.axvline(np.mean(tokens), color="red", linestyle="--", label="Mean tokens")
plt.xlabel("Tokens per chunk")
plt.ylabel("Count")
plt.title("Chunk token length distribution")
plt.legend()
plt.grid(alpha=0.3, linestyle=":")
plt.show()